# Iteratives Verhältniswahlrecht – Demonstration

Dieses Notebook demonstriert das iterative Wahlverfahren anhand kleiner, per Taschenrechner nachvollziehbarer Zahlen.
Es zeigt ausschließlich den Wahlablauf – keine Sitzverteilung oder Wahlkreiszuordnung.

Konzepterklärung: [iterative_verhältniswahl.md](../../docs/source/de/iterative_verhältniswahl.md)

## Setup

**Grundprinzip der Beispiele:** 1 Wahlkreis, 100 Stimmen insgesamt → Stimmanzahl = Prozentzahl.

Normalerweise generiert die Simulation Stimmen zufällig. In diesem Notebook werden stattdessen feste Stimmen injiziert, damit die Beispiele am Taschenrechner nachvollziehbar sind:

- Erster Wahlgang: `election.start(votes={'A': 28, ...})`
- Folgerunden: `runde.getNextRoundInput().with_votes({'A': 35, ...})`

In [ ]:
import pandas as pd

from ipres import (
    Election, ElectionConfig, ElectionRound,
    Ballot, DrawOfLots, DrawLotsStrategy,
    SuperMajorityMargin, MarginUnit,
)
from ipres.constituencies_config import ConstituenciesConfig

# 1 Wahlkreis, 100 Stimmen
cc = ConstituenciesConfig.from_dataframe(pd.DataFrame({
    'constituency_name': ['WK'],
    'constituency_size': [100],
    'turnout_percent':   [100.0],
}))

---
## Beispiel 1: Reduktionsverfahren (3 Wahlgänge)

Fünf Parteien treten an. Gewinnerschwelle: **52 %** (Mehrheitsüberschuss 2 %).

| Partei | Wahlgang 1 | Kumuliert | Wahlgang 2 | Kumuliert | Wahlgang 3 |
|--------|:----------:|:---------:|:----------:|:---------:|:----------:|
| A      | 28         | 28 %      | 35         | 35 %      | **52 ✓**   |
| B      | 25         | 53 %      | 33         | 68 % ≥ ⅔  | 48         |
| C      | 20         | 73 % ≥ ⅔  | 32         | —         | —          |
| D      | 15         | —         | —          | —         | —          |
| E      | 12         | —         | —          | —         | —          |

*Reduktionsregel:* Die Parteien werden absteigend nach Stimmenanteil summiert, bis ⅔ aller Stimmen erreicht sind.
Alle Parteien, die in dieser Summe nicht mehr enthalten sind, scheiden aus.

In [ ]:
config = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B', 'C', 'D', 'E'],
    parliament_majority_margin=SuperMajorityMargin(2.0, MarginUnit.PERCENT),
    draw_lots_strategy=DrawLotsStrategy.MARGINAL_LEAD,
)

election = Election(electionConfig=config)

# --- Wahlgang 1 ---
runde1 = election.start(votes={'A': 28, 'B': 25, 'C': 20, 'D': 15, 'E': 12})

print("=== Wahlgang 1 ===")
print(f"Schwelle: {config.getParliamentMajorityPercent():.0f} %")
print(runde1.getContestantsVotesAfterPossibleCoalitions().to_string())
print(f"\nGewinner: {runde1.getWinner()}")
print(f"Nächste Runde: {runde1.hasNext()}")
print(f"Noch im Rennen: {list(runde1.getNextRoundInput().contestants.keys())}")
print("  → D (15%) und E (12%) ausgeschieden: A+B+C = 73% ≥ 66,7% (⅔)")

In [ ]:
# --- Wahlgang 2 (A, B, C) ---
runde2 = election.runNextIteration(
    iterationInput=runde1.getNextRoundInput().with_votes({'A': 35, 'B': 33, 'C': 32})
)

print("=== Wahlgang 2 ===")
print(runde2.getContestantsVotesAfterPossibleCoalitions().to_string())
print(f"\nGewinner: {runde2.getWinner()}")
print(f"Nächste Runde: {runde2.hasNext()}")
print(f"Noch im Rennen: {list(runde2.getNextRoundInput().contestants.keys())}")
print("  → C (32%) ausgeschieden: A+B = 68% ≥ 66,7% (⅔)")

In [ ]:
# --- Wahlgang 3 (A, B) ---
runde3 = election.runNextIteration(
    iterationInput=runde2.getNextRoundInput().with_votes({'A': 52, 'B': 48})
)

print("=== Wahlgang 3 ===")
print(runde3.getContestantsVotesAfterPossibleCoalitions().to_string())
print(f"\nGewinner: {runde3.getWinner().name}")
print(f"  → A erreicht 52% ≥ 52% (Schwelle) → Wahl beendet nach {election.getNumberOfIterations()} Runden")

### Automatischer Ablauf

`Election.run()` führt alle Runden automatisch durch. Mit zufälligen Stimmen und festem `seed` ist das Ergebnis reproduzierbar:

In [ ]:
config_auto = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B', 'C', 'D', 'E'],
    parliament_majority_margin=SuperMajorityMargin(2.0, MarginUnit.PERCENT),
    draw_lots_strategy=DrawLotsStrategy.MARGINAL_LEAD,
    seed=47,
)

election_auto = Election(electionConfig=config_auto)
final = election_auto.run()

print(f"Gewinner: {final.getWinner().name}  ({election_auto.getNumberOfIterations()} Runden)")
for i, r in enumerate(election_auto.iterations, 1):
    votes = r.getContestantsVotesAfterPossibleCoalitions()
    print(f"  Runde {i}: {votes.to_dict()}")

---
## Beispiel 2: Koalitionsbildung

Drei Parteien, Gewinnerschwelle **55 %**. Keine Partei erreicht die Schwelle allein.
Nach dem ersten Wahlgang bilden B und C eine Koalition.

| Partei | Stimmen | Anteil |
|--------|:-------:|:------:|
| A      | 40      | 40 %   |
| B      | 35      | 35 %   |
| C      | 25      | 25 %   |

B + C zusammen: **60 % ≥ 55 %** → Koalition gewinnt sofort, kein zweiter Wahlgang nötig.

In [ ]:
config_k = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B', 'C'],
    parliament_majority_margin=SuperMajorityMargin(5.0, MarginUnit.PERCENT),
    draw_lots_strategy=DrawLotsStrategy.MARGINAL_LEAD,
)

election_k = Election(electionConfig=config_k)
runde_k = election_k.start(votes={'A': 40, 'B': 35, 'C': 25})

print(f"Schwelle: {config_k.getParliamentMajorityPercent():.0f} %")
print(runde_k.getContestantsVotesAfterPossibleCoalitions().to_string())
print(f"\nGewinner vor Koalition: {runde_k.getWinner()}")

# Koalition B+C bilden
runde_k.formCoalition("B+C", ["B", "C"])

print(f"\nNach Koalitionsbildung B+C (35+25=60%):")
print(runde_k.getContestantsVotesAfterPossibleCoalitions().to_string())
print(f"\nGewinner: {runde_k.getWinner().name}")
print(f"Runden gesamt: {election_k.getNumberOfIterations()}")

---
## Beispiel 3: Sonderfall Losentscheid

Zwei Parteien, Gewinnerschwelle **52 %**. A erhält in beiden Wahlgängen 51 Stimmen – nie genug zum Gewinnen, keine Reduktion möglich.
Nach zwei ergebnislosen Runden mit denselben zwei Parteien entscheidet im dritten Durchgang das Los.

Mit `DrawLotsStrategy.MARGINAL_LEAD` gilt der marginale Stimmunterschied als zufällig entstanden — die Partei, die zufällig etwas mehr Stimmen erhalten hat (hier A mit 51 > 49), gewinnt.

In [ ]:
config_l = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B'],
    parliament_majority_margin=SuperMajorityMargin(2.0, MarginUnit.PERCENT),
    draw_lots_strategy=DrawLotsStrategy.MARGINAL_LEAD,
)

election_l = Election(electionConfig=config_l)

stimmen_ab = {'A': 51, 'B': 49}   # A < 52% → kein Gewinner

# Runde 1
r1_l = election_l.start(votes=stimmen_ab)

# Runde 2
r2_l = election_l.runNextIteration(
    iterationInput=r1_l.getNextRoundInput().with_votes(stimmen_ab)
)

# Runde 3 – DrawOfLots wird automatisch ausgelöst
r3_l = election_l.runNextIteration(
    iterationInput=r2_l.getNextRoundInput().with_votes(stimmen_ab)
)

for i, r in enumerate([r1_l, r2_l, r3_l], 1):
    art = type(r).__name__
    print(f"Runde {i} ({art}): Gewinner={r.getWinner()}")

print(f"\nLosentscheid ausgelöst: {isinstance(r3_l, DrawOfLots)}")
print(f"Gewinner: {r3_l.getWinner().name}  (hat mit {stimmen_ab['A']} > {stimmen_ab['B']} die meisten Stimmen)")

---

## `DrawLotsStrategy` — Beide Varianten im Vergleich

Wenn zwei Parteien in **zwei aufeinanderfolgenden Wahlgängen** keinen Gewinner produzieren, entscheidet im **dritten Durchgang** das Los.

- **`RANDOM`** *(Standard)*: gleichverteiltes Zufallsverfahren — der Gewinner wird per Zufallszug bestimmt.
- **`MARGINAL_LEAD`**: Der marginale Stimmunterschied gilt als zufällig entstanden — die Partei, die zufällig etwas mehr Stimmen erhalten hat, gewinnt.

In [ ]:
stimmen = {'A': 51, 'B': 49}   # A = 51 % < 52 % → kein Gewinner

for strategy in [DrawLotsStrategy.RANDOM, DrawLotsStrategy.MARGINAL_LEAD]:
    cfg = ElectionConfig(
        constituencies_config=cc,
        participating_parties=['A', 'B'],
        ballot_majority_margin=SuperMajorityMargin(2.0, MarginUnit.PERCENT),
        parliament_majority_margin=SuperMajorityMargin(2.0, MarginUnit.PERCENT),
        draw_lots_strategy=strategy,
        seed=7,
    )
    e = Election(electionConfig=cfg)
    r1 = e.start(votes=stimmen)                                           # Runde 1: kein Gewinner
    r2 = e.runNextIteration(r1.getNextRoundInput().with_votes(stimmen))   # Runde 2: kein Gewinner → Losentscheid
    r3 = e.runNextIteration(r2.getNextRoundInput().with_votes(stimmen))   # Runde 3: DrawOfLots
    print(f"{strategy.name:26s} → Gewinner: {e.getWinner().name}  {'(durch Los)' if r3.wasDecidedByLot() else ''}")

---

## `ballot_majority_margin` — Wahlgangschwelle

Der `ballot_majority_margin` legt fest, welchen Stimmenanteil ein Kandidat in einem **einzelnen Wahlgang** übertreffen muss, um diesen Wahlgang zu gewinnen. Er ist unabhängig von der `parliament_majority_margin`, die den Regierungsmehrheitsanteil im Parlament bestimmt.

Standard: `SuperMajorityMargin(2.0, MarginUnit.PERCENT)` → Schwelle 52 %. Die Angabe ist auch in Sitzen möglich (`MarginUnit.SEATS`), was sich bei größeren Konfigurationen anbietet (vgl. Notebook `globale_konfiguration`).

In [ ]:
# A erhält 53 % der Stimmen.

# ballot_majority_margin = 2 % (Standard): Schwelle 52 % → A gewinnt sofort
config_bm_low = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B'],
    ballot_majority_margin=SuperMajorityMargin(2.0, MarginUnit.PERCENT),
    parliament_majority_margin=SuperMajorityMargin(2.0, MarginUnit.PERCENT),
)
r = Election(electionConfig=config_bm_low).start(votes={'A': 53, 'B': 47})
print(f"ballot_majority_margin=2%  →  Schwelle {config_bm_low.getBallotMajorityPercent():.0f}%  →  A=53%  →  Gewinner: {r.getWinner()}")

# ballot_majority_margin = 5 % (höher): Schwelle 55 % → A reicht nicht aus
config_bm_high = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B'],
    ballot_majority_margin=SuperMajorityMargin(5.0, MarginUnit.PERCENT),
    parliament_majority_margin=SuperMajorityMargin(5.0, MarginUnit.PERCENT),
)
r2 = Election(electionConfig=config_bm_high).start(votes={'A': 53, 'B': 47})
w = r2.getWinner()
print(f"ballot_majority_margin=5%  →  Schwelle {config_bm_high.getBallotMajorityPercent():.0f}%  →  A=53%  →  Gewinner: {w if w else 'kein Gewinner → nächste Runde nötig'}")

---

## Stimmeninjektion mit mehreren Wahlkreisen

Bei mehreren Wahlkreisen wird das `votes`-Argument als verschachteltes Dict übergeben:
`{wahlkreis: {partei: stimmen}}`. Die Stimmen werden über alle Wahlkreise summiert, um die Gesamtstimmen je Partei zu erhalten.

In [ ]:
# 2-Wahlkreis-Konfiguration
cc2 = ConstituenciesConfig.from_dataframe(pd.DataFrame({
    'constituency_name': ['WK1', 'WK2'],
    'constituency_size': [100, 100],
    'turnout_percent':   [100.0, 100.0],
}))

config_mc = ElectionConfig(
    constituencies_config=cc2,
    participating_parties=['A', 'B', 'C'],
    parliament_majority_margin=SuperMajorityMargin(5.0, MarginUnit.PERCENT),
)

election_mc = Election(electionConfig=config_mc)
runde_mc = election_mc.start(votes={
    'WK1': {'A': 28, 'B': 22, 'C': 50},   # WK1: C führt
    'WK2': {'A': 40, 'B': 30, 'C': 30},   # WK2: A führt
})

print("Stimmen je Wahlkreis:")
print(runde_mc.vote_matrix.getVotes())
votes = runde_mc.getContestantsVotesAfterPossibleCoalitions()
total = votes.sum()
print(f"\nGesamtstimmen:  {votes.to_dict()}")
print(f"Stimmenanteile: { {k: f'{v/total*100:.1f}%' for k, v in votes.items()} }")
w_mc = runde_mc.getWinner()
print(f"Gewinner: {w_mc.name if w_mc else 'kein Gewinner'}")